# Classification d'images CIFAR-10 — ML classiques, CNN et Transfer Learning

**Auteurs :** Yves CHEKOUA & Mohamed Mehdi TRABELSSI
**Promotion :** SN2 — Université de Technologie de Troyes

---
## Plan du notebook
1. Analyse du dataset CIFAR-10
2. Préparation des données pour les algorithmes ML classiques
3. Algorithme ML 1 : SVM
4. Algorithme ML 2 : Régression Logistique
5. Algorithme ML 3 : Random Forest
6. Préparation des données pour les CNN
7. CNN Architecture 1 : CNN Simple (Baseline)
8. CNN Architecture 2 : CNN Intermédiaire
9. CNN Architecture 3 : CNN Avancé (Régularisé)
10. CNN Hybride : Transfer Learning MobileNetV2
11. Comparaison générale
12. Conclusion


## Imports et chargement des bibliothèques

In [ ]:
# ── Bibliothèques de base ──────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
import warnings
warnings.filterwarnings('ignore')

# ── Machine Learning classique ────────────────────────────────────
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix)

# ── Deep Learning ─────────────────────────────────────────────────
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.datasets import cifar10

print(f"TensorFlow : {tf.__version__}")
print(f"GPU disponible : {len(tf.config.list_physical_devices('GPU')) > 0}")


## 1. Analyse du dataset CIFAR-10

In [ ]:
# Chargement du dataset depuis Keras
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

CLASSES = ['avion','voiture','oiseau','chat','cerf',
           'chien','grenouille','cheval','bateau','camion']

print(f"Train : {x_train.shape}  |  Test : {x_test.shape}")
print(f"Valeurs pixels : [{x_train.min()}, {x_train.max()}]")
print(f"Classes : {CLASSES}")


In [ ]:
# Afficher un exemple par classe
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    idx = np.where(y_train.flatten() == i)[0][0]
    ax.imshow(x_train[idx])
    ax.set_title(CLASSES[i], fontsize=10)
    ax.axis('off')
plt.suptitle('Exemples CIFAR-10 – une image par classe', fontsize=13)
plt.tight_layout()
plt.show()


In [ ]:
# Distribution des classes (dataset parfaitement équilibré)
unique, counts = np.unique(y_train, return_counts=True)
plt.figure(figsize=(9, 3))
sns.barplot(x=[CLASSES[i] for i in unique], y=counts, palette='Blues_d')
plt.title('Distribution des classes – CIFAR-10 (5 000 images/classe)')
plt.ylabel("Nombre d'images")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()
print("Dataset parfaitement équilibré : pas de biais de classe.")


## 2. Préparation des données pour les algorithmes ML classiques
Les images 32×32×3 sont **aplaties** en vecteurs de 3 072 features et **normalisées** dans [0, 1].
On utilise un sous-ensemble de **10 000 images** pour la recherche d'hyperparamètres.


In [ ]:
# Aplatissement : 32x32x3 → vecteur de 3072 features
x_train_flat = x_train.reshape(50000, -1).astype('float32') / 255.0
x_test_flat  = x_test.reshape(10000,  -1).astype('float32') / 255.0
y_train_flat = y_train.flatten()
y_test_flat  = y_test.flatten()

# Sous-ensemble pour GridSearchCV (limite le temps de calcul)
N_GRID = 10_000
x_tr = x_train_flat[:N_GRID]
y_tr = y_train_flat[:N_GRID]

print(f"Shape features : {x_train_flat.shape}")
print(f"Sous-ensemble GridSearch : {N_GRID} images")


## 3. Algorithme ML 1 : SVM (Support Vector Machine)
**Justification :** Le kernel RBF projette les données dans un espace de dimension supérieure,
rendant possible la séparation non linéaire des classes de pixels bruts.
**Hyperparamètres testés :** C ∈ {0.1, 1, 10}, gamma ∈ {scale, 0.01}


In [ ]:
param_grid_svm = {'C': [0.1, 1, 10], 'gamma': ['scale', 0.01]}

print("GridSearchCV SVM en cours (~60 min sur CPU)...")
t0 = time.time()
grid_svm = GridSearchCV(SVC(kernel='rbf'), param_grid_svm, cv=3, verbose=1, n_jobs=-1)
grid_svm.fit(x_tr, y_tr)
t_svm_grid = time.time() - t0

print(f"Meilleurs paramètres : {grid_svm.best_params_}")
print(f"Temps GridSearch     : {t_svm_grid:.1f} s")


In [ ]:
# Évaluation sur le jeu de test complet
y_pred_svm = grid_svm.predict(x_test_flat)
acc_svm = accuracy_score(y_test_flat, y_pred_svm)

print(f"Exactitude SVM : {acc_svm*100:.2f}%")
print()
print(classification_report(y_test_flat, y_pred_svm, target_names=CLASSES))

cm_svm = confusion_matrix(y_test_flat, y_pred_svm)
plt.figure(figsize=(9, 7))
sns.heatmap(cm_svm, annot=True, fmt='d', cmap='Purples',
            xticklabels=CLASSES, yticklabels=CLASSES)
plt.title(f'SVM – Matrice de confusion (acc={acc_svm*100:.1f}%)')
plt.xlabel('Prédit')
plt.ylabel('Réel')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


## 4. Algorithme ML 2 : Régression Logistique
**Justification :** Algorithme linéaire rapide utilisé comme référence (baseline).
Malgré sa simplicité, il peut capturer certaines structures dans l'espace des features normalisées.
**Hyperparamètres testés :** C ∈ {0.01, 0.1, 1, 10}


In [ ]:
param_grid_lr = {'C': [0.01, 0.1, 1, 10]}

print("GridSearchCV Régression Logistique (~10 min)...")
t0 = time.time()
grid_lr = GridSearchCV(
    LogisticRegression(max_iter=500, solver='saga'),
    param_grid_lr, cv=3, verbose=1, n_jobs=-1
)
grid_lr.fit(x_tr, y_tr)
t_lr = time.time() - t0

print(f"Meilleurs paramètres : {grid_lr.best_params_}")
print(f"Temps GridSearch     : {t_lr:.1f} s")


In [ ]:
y_pred_lr = grid_lr.predict(x_test_flat)
acc_lr = accuracy_score(y_test_flat, y_pred_lr)

print(f"Exactitude Régression Logistique : {acc_lr*100:.2f}%")
print()
print(classification_report(y_test_flat, y_pred_lr, target_names=CLASSES))

cm_lr = confusion_matrix(y_test_flat, y_pred_lr)
plt.figure(figsize=(9, 7))
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Oranges',
            xticklabels=CLASSES, yticklabels=CLASSES)
plt.title(f'Régression Logistique – Matrice de confusion (acc={acc_lr*100:.1f}%)')
plt.xlabel('Prédit')
plt.ylabel('Réel')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


## 5. Algorithme ML 3 : Random Forest
**Justification :** Algorithme d'ensemble (bagging) combinant plusieurs arbres de décision.
Plus rapide que le SVM et naturellement résistant au surapprentissage.
**GridSearchCV sur 10 000 images**, entraînement final sur **50 000 images**.


In [ ]:
param_grid_rf = {
    'n_estimators': [100, 200],
    'max_depth':    [20, 30, None],
    'max_features': ['sqrt', 'log2']
}

print("GridSearchCV Random Forest (~15 min)...")
t0 = time.time()
grid_rf = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_grid_rf, cv=3, scoring='accuracy', verbose=1, n_jobs=-1
)
grid_rf.fit(x_tr, y_tr)
t_rf_grid = time.time() - t0

print(f"Meilleurs paramètres : {grid_rf.best_params_}")
print(f"Temps GridSearch     : {t_rf_grid:.1f} s")
print(f"Meilleure exactitude : {grid_rf.best_score_*100:.2f}%")


In [ ]:
# Entraînement final sur les 50 000 images
rf = RandomForestClassifier(**grid_rf.best_params_, random_state=42, n_jobs=-1)

print("Entraînement final sur 50 000 images...")
t0 = time.time()
rf.fit(x_train_flat, y_train_flat)
t_rf_train = time.time() - t0
print(f"Terminé en {t_rf_train:.1f} s")


In [ ]:
y_pred_rf = rf.predict(x_test_flat)
acc_rf       = accuracy_score(y_test_flat, y_pred_rf)
acc_rf_train = accuracy_score(y_train_flat, rf.predict(x_train_flat))

print(f"Exactitude entraînement : {acc_rf_train*100:.2f}%  (surapprentissage visible)")
print(f"Exactitude test         : {acc_rf*100:.2f}%")
print()
print(classification_report(y_test_flat, y_pred_rf, target_names=CLASSES))

cm_rf = confusion_matrix(y_test_flat, y_pred_rf)
plt.figure(figsize=(9, 7))
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASSES, yticklabels=CLASSES)
plt.title(f'Random Forest – Matrice de confusion (acc={acc_rf*100:.1f}%)')
plt.xlabel('Prédit')
plt.ylabel('Réel')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


In [ ]:
# Carte d'importance des pixels par canal RGB
importances = rf.feature_importances_.reshape(32, 32, 3)
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for i, ax in enumerate(axes):
    im = ax.imshow(importances[:, :, i], cmap='hot')
    ax.set_title(['Canal Rouge (R)', 'Canal Vert (G)', 'Canal Bleu (B)'][i])
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.suptitle('Importance des pixels par canal RGB – Random Forest')
plt.tight_layout()
plt.show()


## 6. Préparation des données pour les CNN
Les images conservent leur format **32×32×3**. Normalisation dans [0, 1].
On utilise les **50 000 images** pour les CNN from scratch.


In [ ]:
# Normalisation pour CNN : valeurs dans [0, 1]
x_train_cnn_raw = x_train.astype('float32') / 255.0
x_test_cnn_raw  = x_test.astype('float32')  / 255.0
y_train_cat = tf.keras.utils.to_categorical(y_train, 10)
y_test_cat  = tf.keras.utils.to_categorical(y_test,  10)

# Hyperparamètres communs aux 3 architectures CNN
BATCH_SIZE = 64
EPOCHS_CNN = 15

print(f"Shape train CNN : {x_train_cnn_raw.shape}")
print(f"Shape labels    : {y_train_cat.shape}")


## 7. CNN Architecture 1 : CNN Simple (Baseline)
Architecture minimale pour établir une performance de référence.
`Conv2D(32) → MaxPool → Conv2D(64) → MaxPool → Dense(64) → Dense(10, softmax)`


In [ ]:
def build_cnn_simple(input_shape=(32, 32, 3), num_classes=10):
    model = models.Sequential([
        # Bloc 1 : extraction de features basse résolution
        layers.Conv2D(32, (3,3), activation='relu', padding='same', input_shape=input_shape),
        layers.MaxPooling2D(2, 2),
        # Bloc 2 : features plus abstraites
        layers.Conv2D(64, (3,3), activation='relu', padding='same'),
        layers.MaxPooling2D(2, 2),
        # Tête de classification
        layers.Flatten(),
        layers.Dense(64, activation='relu'),
        layers.Dense(num_classes, activation='softmax')
    ], name='CNN_Simple')
    return model

cnn1 = build_cnn_simple()
cnn1.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
cnn1.summary()


In [ ]:
cb_cnn = [
    EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1)
]

print("Entraînement CNN Simple...")
t0 = time.time()
hist_cnn1 = cnn1.fit(
    x_train_cnn_raw, y_train_cat,
    epochs=EPOCHS_CNN, batch_size=BATCH_SIZE,
    validation_split=0.1, callbacks=cb_cnn, verbose=1
)
t_cnn1 = time.time() - t0
print(f"Temps : {t_cnn1:.1f} s")


In [ ]:
_, acc_cnn1 = cnn1.evaluate(x_test_cnn_raw, y_test_cat, verbose=0)
y_pred_cnn1 = np.argmax(cnn1.predict(x_test_cnn_raw, verbose=0), axis=1)

print(f"CNN Simple – Exactitude test : {acc_cnn1*100:.2f}%  |  Temps : {t_cnn1:.1f} s")
print()
print(classification_report(y_test_flat, y_pred_cnn1, target_names=CLASSES))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(hist_cnn1.history['accuracy'],     label='Train')
axes[0].plot(hist_cnn1.history['val_accuracy'], label='Validation')
axes[0].set_title('CNN Simple – Exactitude')
axes[0].set_xlabel('Époque'); axes[0].legend(); axes[0].grid(True)
axes[1].plot(hist_cnn1.history['loss'],     label='Train')
axes[1].plot(hist_cnn1.history['val_loss'], label='Validation')
axes[1].set_title('CNN Simple – Perte')
axes[1].set_xlabel('Époque'); axes[1].legend(); axes[1].grid(True)
plt.tight_layout(); plt.show()


## 8. CNN Architecture 2 : CNN Intermédiaire
Double convolution par bloc pour capturer des features plus riches.
`Conv2D(32)×2 → MaxPool → Conv2D(64)×2 → MaxPool → Dense(128) → Dense(10, softmax)`


In [ ]:
def build_cnn_intermediaire(input_shape=(32, 32, 3), num_classes=10):
    model = models.Sequential([
        # Bloc 1 : double convolution 32 filtres
        layers.Conv2D(32, (3,3), activation='relu', padding='same', input_shape=input_shape),
        layers.Conv2D(32, (3,3), activation='relu', padding='same'),
        layers.MaxPooling2D(2, 2),
        # Bloc 2 : double convolution 64 filtres
        layers.Conv2D(64, (3,3), activation='relu', padding='same'),
        layers.Conv2D(64, (3,3), activation='relu', padding='same'),
        layers.MaxPooling2D(2, 2),
        # Tête de classification plus large
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dense(num_classes, activation='softmax')
    ], name='CNN_Intermediaire')
    return model

cnn2 = build_cnn_intermediaire()
cnn2.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
cnn2.summary()


In [ ]:
print("Entraînement CNN Intermédiaire...")
t0 = time.time()
hist_cnn2 = cnn2.fit(
    x_train_cnn_raw, y_train_cat,
    epochs=EPOCHS_CNN, batch_size=BATCH_SIZE,
    validation_split=0.1, callbacks=cb_cnn, verbose=1
)
t_cnn2 = time.time() - t0
print(f"Temps : {t_cnn2:.1f} s")


In [ ]:
_, acc_cnn2 = cnn2.evaluate(x_test_cnn_raw, y_test_cat, verbose=0)
y_pred_cnn2 = np.argmax(cnn2.predict(x_test_cnn_raw, verbose=0), axis=1)

print(f"CNN Intermédiaire – Exactitude test : {acc_cnn2*100:.2f}%  |  Temps : {t_cnn2:.1f} s")
print()
print(classification_report(y_test_flat, y_pred_cnn2, target_names=CLASSES))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(hist_cnn2.history['accuracy'],     label='Train')
axes[0].plot(hist_cnn2.history['val_accuracy'], label='Validation')
axes[0].set_title('CNN Intermédiaire – Exactitude')
axes[0].set_xlabel('Époque'); axes[0].legend(); axes[0].grid(True)
axes[1].plot(hist_cnn2.history['loss'],     label='Train')
axes[1].plot(hist_cnn2.history['val_loss'], label='Validation')
axes[1].set_title('CNN Intermédiaire – Perte')
axes[1].set_xlabel('Époque'); axes[1].legend(); axes[1].grid(True)
plt.tight_layout(); plt.show()


## 9. CNN Architecture 3 : CNN Avancé (Régularisé)
Ajout de **BatchNormalization** et **Dropout** pour réduire le surapprentissage.
`Conv2D(32)×2 + BN + Dropout(0.2) → Conv2D(64)×2 + BN + Dropout(0.3) → Dense(128) + BN + Dropout(0.4) → Dense(10)`


In [ ]:
def build_cnn_avance(input_shape=(32, 32, 3), num_classes=10):
    model = models.Sequential([
        # Bloc 1 : convolutions + régularisation
        layers.Conv2D(32, (3,3), padding='same', input_shape=input_shape),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Conv2D(32, (3,3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D(2, 2),
        layers.Dropout(0.2),
        # Bloc 2 : features plus profondes + régularisation renforcée
        layers.Conv2D(64, (3,3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Conv2D(64, (3,3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D(2, 2),
        layers.Dropout(0.3),
        # Tête de classification régularisée
        layers.Flatten(),
        layers.Dense(128),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Dropout(0.4),
        layers.Dense(num_classes, activation='softmax')
    ], name='CNN_Avance_Regularise')
    return model

cnn3 = build_cnn_avance()
cnn3.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
cnn3.summary()


In [ ]:
print("Entraînement CNN Avancé (régularisé)...")
t0 = time.time()
hist_cnn3 = cnn3.fit(
    x_train_cnn_raw, y_train_cat,
    epochs=EPOCHS_CNN, batch_size=BATCH_SIZE,
    validation_split=0.1, callbacks=cb_cnn, verbose=1
)
t_cnn3 = time.time() - t0
print(f"Temps : {t_cnn3:.1f} s")


In [ ]:
_, acc_cnn3 = cnn3.evaluate(x_test_cnn_raw, y_test_cat, verbose=0)
y_pred_cnn3 = np.argmax(cnn3.predict(x_test_cnn_raw, verbose=0), axis=1)

print(f"CNN Avancé – Exactitude test : {acc_cnn3*100:.2f}%  |  Temps : {t_cnn3:.1f} s")
print()
print(classification_report(y_test_flat, y_pred_cnn3, target_names=CLASSES))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(hist_cnn3.history['accuracy'],     label='Train')
axes[0].plot(hist_cnn3.history['val_accuracy'], label='Validation')
axes[0].set_title('CNN Avancé – Exactitude')
axes[0].set_xlabel('Époque'); axes[0].legend(); axes[0].grid(True)
axes[1].plot(hist_cnn3.history['loss'],     label='Train')
axes[1].plot(hist_cnn3.history['val_loss'], label='Validation')
axes[1].set_title('CNN Avancé – Perte')
axes[1].set_xlabel('Époque'); axes[1].legend(); axes[1].grid(True)
plt.tight_layout(); plt.show()


## 10. CNN Hybride : Transfer Learning MobileNetV2
**Approche en deux phases :**
- Phase 1 : couches de la base gelées → seule la tête FC est entraînée (`lr=1e-3`)
- Phase 2 : fine-tuning des 30 dernières couches (`lr=1e-5`)

> **Pourquoi 96×96 ?**
> MobileNetV2 applique 5 MaxPooling × 2. Sur 32×32, les feature maps tombent à 1×1 →
> GlobalAveragePooling ne contient aucune information spatiale → accuracy de seulement 49%.
> À 96×96 les feature maps font 3×3, suffisant pour extraire des features utiles.


In [ ]:
IMG_SIZE = 96
N_CNN_TR = 10_000  # sous-ensemble pour accélérer
N_CNN_TE = 2_000

print(f"Redimensionnement {N_CNN_TR} images 32×32 → {IMG_SIZE}×{IMG_SIZE}...")
t0 = time.time()

x_cnn_tr = tf.image.resize(x_train[:N_CNN_TR], [IMG_SIZE, IMG_SIZE]).numpy()
x_cnn_te = tf.image.resize(x_test[:N_CNN_TE],  [IMG_SIZE, IMG_SIZE]).numpy()

# Normalisation MobileNetV2 : valeurs dans [-1, 1]
x_cnn_tr = x_cnn_tr.astype('float32') / 127.5 - 1.0
x_cnn_te = x_cnn_te.astype('float32') / 127.5 - 1.0

y_cnn_tr = y_train[:N_CNN_TR].flatten()
y_cnn_te = y_test[:N_CNN_TE].flatten()

print(f"Terminé en {time.time()-t0:.1f} s")
print(f"Train : {x_cnn_tr.shape}  |  Min/Max : {x_cnn_tr.min():.1f} / {x_cnn_tr.max():.1f}")


In [ ]:
# Chargement de la base pré-entraînée (ImageNet), sans tête de classification
base = MobileNetV2(weights='imagenet', include_top=False,
                   input_shape=(IMG_SIZE, IMG_SIZE, 3))

# Geler toutes les couches (Phase 1)
for layer in base.layers:
    layer.trainable = False

# Construction de la nouvelle tête FC
x   = base.output
x   = layers.GlobalAveragePooling2D()(x)
x   = layers.Dense(128, activation='relu')(x)
x   = layers.Dropout(0.3)(x)
out = layers.Dense(10, activation='softmax')(x)

model_hybrid = models.Model(inputs=base.input, outputs=out, name='CNN_Hybride_MobileNetV2')

print(f"Paramètres totaux  : {model_hybrid.count_params():,}")
print(f"Paramètres gelés   : {base.count_params():,}")
print(f"Paramètres entraîn.: {model_hybrid.count_params() - base.count_params():,}")


In [ ]:
# ── Phase 1 : Freeze ──────────────────────────────────────────────
model_hybrid.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
cb_hybrid = [
    EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1)
]

print("Phase 1 – Freeze : entraînement de la tête FC uniquement...")
t0 = time.time()
hist_freeze = model_hybrid.fit(
    x_cnn_tr, y_cnn_tr,
    epochs=20, batch_size=64, validation_split=0.1,
    callbacks=cb_hybrid, verbose=1
)
t_freeze = time.time() - t0
print(f"Phase 1 terminée en {t_freeze/60:.1f} min")


In [ ]:
# ── Phase 2 : Fine-tuning ─────────────────────────────────────────
# Dégeler les 30 dernières couches de la base
for layer in base.layers[-30:]:
    layer.trainable = True

model_hybrid.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),  # lr très faible pour ne pas écraser les poids
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("Phase 2 – Fine-tuning : 30 dernières couches dégelées...")
t0 = time.time()
hist_ft = model_hybrid.fit(
    x_cnn_tr, y_cnn_tr,
    epochs=10, batch_size=64, validation_split=0.1,
    callbacks=cb_hybrid, verbose=1
)
t_ft = time.time() - t0
t_hybrid_total = t_freeze + t_ft
print(f"Phase 2 terminée en {t_ft/60:.1f} min")
print(f"Temps total CNN Hybride : {t_hybrid_total/60:.1f} min")


In [ ]:
# Courbes d'apprentissage (Phase 1 + Phase 2 concaténées)
acc_all  = hist_freeze.history['accuracy']    + hist_ft.history['accuracy']
val_all  = hist_freeze.history['val_accuracy']+ hist_ft.history['val_accuracy']
ep_split = len(hist_freeze.history['accuracy'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, tr, vl, title in zip(axes,
    [acc_all, hist_freeze.history['loss']+hist_ft.history['loss']],
    [val_all,  hist_freeze.history['val_loss']+hist_ft.history['val_loss']],
    ['Exactitude', 'Perte']):
    ax.plot(tr, label='Train', linewidth=2)
    ax.plot(vl, label='Validation', linewidth=2)
    ax.axvline(ep_split-1, color='gray', linestyle='--', label='Début fine-tuning')
    ax.set_title(f'CNN Hybride MobileNetV2 – {title}')
    ax.set_xlabel('Époque'); ax.legend(); ax.grid(True)
plt.tight_layout(); plt.show()


In [ ]:
_, acc_hybrid = model_hybrid.evaluate(x_cnn_te, y_cnn_te, verbose=0)
y_pred_hybrid = np.argmax(model_hybrid.predict(x_cnn_te, verbose=0), axis=1)

print(f"CNN Hybride MobileNetV2 – Exactitude test : {acc_hybrid*100:.2f}%")
print(f"Temps total entraînement : {t_hybrid_total/60:.1f} min")
print()
print(classification_report(y_cnn_te, y_pred_hybrid, target_names=CLASSES))

cm_hybrid = confusion_matrix(y_cnn_te, y_pred_hybrid)
plt.figure(figsize=(9, 7))
sns.heatmap(cm_hybrid, annot=True, fmt='d', cmap='Greens',
            xticklabels=CLASSES, yticklabels=CLASSES)
plt.title(f'CNN Hybride MobileNetV2 – Matrice de confusion (acc={acc_hybrid*100:.1f}%)')
plt.xlabel('Prédit'); plt.ylabel('Réel')
plt.xticks(rotation=45, ha='right')
plt.tight_layout(); plt.show()


## 11. Comparaison générale

In [ ]:
# Tableau récapitulatif
modeles = ['SVM (RBF)', 'Régression Logistique', 'Random Forest',
           'CNN Simple', 'CNN Intermédiaire', 'CNN Avancé', 'CNN Hybride (MobileNetV2)']
accs = [acc_svm, acc_lr, acc_rf, acc_cnn1, acc_cnn2, acc_cnn3, acc_hybrid]

print(f"{'Modèle':<30} {'Exactitude test':>16}")
print("─" * 48)
for m, a in zip(modeles, accs):
    print(f"{m:<30} {a*100:>15.2f}%")

# Graphique comparatif
plt.figure(figsize=(10, 5))
colors = ['#d62728','#ff7f0e','#1f77b4','#2ca02c','#9467bd','#8c564b','#17becf']
bars = plt.barh(modeles, [a*100 for a in accs], color=colors)
plt.xlabel('Exactitude test (%)')
plt.title('Comparaison des algorithmes – CIFAR-10')
for bar, a in zip(bars, accs):
    plt.text(bar.get_width()+0.3, bar.get_y()+bar.get_height()/2,
             f'{a*100:.1f}%', va='center', fontweight='bold')
plt.xlim(0, 100)
plt.tight_layout()
plt.show()


---
## 12. Conclusion

| Modèle | Exactitude | Surapprentissage |
|--------|-----------|-----------------|
| SVM (RBF) | ~47,6% | – |
| Régression Logistique | ~40% | – |
| Random Forest | ~47,7% | Fort (99,8% train) |
| CNN Simple | ~70,9% | Modéré |
| CNN Intermédiaire | ~72,5% | Fort |
| CNN Avancé (Régularisé) | ~81,7% | Faible |
| **CNN Hybride MobileNetV2** | **~87,3%** | **Faible** |

### Enseignements

- Le **Transfer Learning** (MobileNetV2) surpasse systématiquement les CNN entraînés de zéro, même avec peu de données : +5,6 points par rapport au meilleur CNN « from scratch »
- La **régularisation** (BatchNormalization + Dropout) apporte un gain de +9 points d'exactitude par rapport à une architecture CNN équivalente sans régularisation
- Les algorithmes ML classiques (SVM, Random Forest) plafonnent autour de 48% sur des pixels bruts aplatis : ils ne capturent pas la structure spatiale des images
- Redimensionner les images à 96×96 (au lieu de 32×32) est indispensable pour exploiter MobileNetV2 sans effondrement des feature maps
- Le Random Forest présente un surapprentissage marqué (99,8% en train contre 47,7% en test), révélant la limite de la représentation en pixels bruts
